In [1]:

import pandas as pd
from google.colab import files

uploaded = files.upload()


Saving bank-additional-full.csv to bank-additional-full.csv


In [2]:
import io

file_name = "bank-additional-full.csv"

df = pd.read_csv(
    io.BytesIO(uploaded[file_name]),
    sep=";"
)

print("Dataset shape:", df.shape)
display(df.head())
print(df.info())


Dataset shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [3]:
# 1. Column names
print("Columns:")
print(df.columns.tolist())

# 2. Missing values
print("\nMissing values:")
print(df.isnull().sum())

# 3. Duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# 4. Target column distribution
print("\nCampaign outcomes:")
print(df["y"].value_counts())

# 5. Data types
print("\nData types:")
print(df.dtypes)


Columns:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Missing values:
age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

Duplicate rows: 12

Campaign outcomes:
y
no     36548
yes     4640
Name: count, dtype: int64

Data types:
age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact

In [4]:
# Keep the original dataset unchanged
df_clean = df.drop_duplicates().copy()

print("Original rows:", len(df))
print("Rows after removing duplicates:", len(df_clean))
print("Duplicate rows removed:", len(df) - len(df_clean))

print("\nUpdated campaign outcomes:")
print(df_clean["y"].value_counts())

print("\nUpdated conversion rate:")
conversion_rate = (df_clean["y"].eq("yes").mean()) * 100
print(f"{conversion_rate:.2f}%")


Original rows: 41188
Rows after removing duplicates: 41176
Duplicate rows removed: 12

Updated campaign outcomes:
y
no     36537
yes     4639
Name: count, dtype: int64

Updated conversion rate:
11.27%


In [5]:
# Conversion rate by contact channel
channel_analysis = (
    df_clean.groupby("contact")
    .agg(
        Total_Contacts=("y", "size"),
        Conversions=("y", lambda x: (x == "yes").sum())
    )
    .reset_index()
)

channel_analysis["Conversion_Rate_%"] = (
    channel_analysis["Conversions"]
    / channel_analysis["Total_Contacts"] * 100
).round(2)

print("CONVERSION BY CONTACT CHANNEL")
display(channel_analysis.sort_values("Conversion_Rate_%", ascending=False))


# Conversion rate by previous campaign outcome
outcome_analysis = (
    df_clean.groupby("poutcome")
    .agg(
        Total_Contacts=("y", "size"),
        Conversions=("y", lambda x: (x == "yes").sum())
    )
    .reset_index()
)

outcome_analysis["Conversion_Rate_%"] = (
    outcome_analysis["Conversions"]
    / outcome_analysis["Total_Contacts"] * 100
).round(2)

print("CONVERSION BY PREVIOUS CAMPAIGN OUTCOME")
display(outcome_analysis.sort_values("Conversion_Rate_%", ascending=False))


CONVERSION BY CONTACT CHANNEL


,contact,Total_Contacts,Conversions,Conversion_Rate_%
0,cellular,26135,3852,14.74
1,telephone,15041,787,5.23


CONVERSION BY PREVIOUS CAMPAIGN OUTCOME


,poutcome,Total_Contacts,Conversions,Conversion_Rate_%
2,success,1373,894,65.11
0,failure,4252,605,14.23
1,nonexistent,35551,3140,8.83


In [6]:
# Conversion by number of contacts in the current campaign
campaign_analysis = (
    df_clean.groupby("campaign")
    .agg(
        Total_Contacts=("y", "size"),
        Conversions=("y", lambda x: (x == "yes").sum())
    )
    .reset_index()
)

campaign_analysis["Conversion_Rate_%"] = (
    campaign_analysis["Conversions"]
    / campaign_analysis["Total_Contacts"] * 100
).round(2)

print("CONVERSION BY CAMPAIGN CONTACT FREQUENCY")
display(campaign_analysis.head(15))


CONVERSION BY CAMPAIGN CONTACT FREQUENCY


,campaign,Total_Contacts,Conversions,Conversion_Rate_%
0,1,17634,2299,13.04
1,2,10568,1211,11.46
2,3,5340,574,10.75
3,4,2650,249,9.40
4,5,1599,120,7.50
5,6,979,75,7.66
6,7,629,38,6.04
7,8,400,17,4.25
8,9,283,17,6.01
9,10,225,12,5.33


In [7]:
# Create customer age groups
age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = [
    "18-25", "26-35", "36-45",
    "46-55", "56-65", "66+"
]

df_clean["age_group"] = pd.cut(
    df_clean["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

age_analysis = (
    df_clean.groupby("age_group", observed=False)
    .agg(
        Total_Contacts=("y", "size"),
        Conversions=("y", lambda x: (x == "yes").sum())
    )
    .reset_index()
)

age_analysis["Conversion_Rate_%"] = (
    age_analysis["Conversions"]
    / age_analysis["Total_Contacts"] * 100
).round(2)

display(age_analysis)


,age_group,Total_Contacts,Conversions,Conversion_Rate_%
0,18-25,1665,349,20.96
1,26-35,14844,1740,11.72
2,36-45,12839,1092,8.51
3,46-55,8247,717,8.69
4,56-65,2963,451,15.22
5,66+,618,290,46.93


In [8]:
month_analysis = (
    df_clean.groupby("month")
    .agg(
        Total_Contacts=("y", "size"),
        Conversions=("y", lambda x: (x == "yes").sum())
    )
    .reset_index()
)

month_analysis["Conversion_Rate_%"] = (
    month_analysis["Conversions"]
    / month_analysis["Total_Contacts"] * 100
).round(2)

display(
    month_analysis.sort_values(
        "Conversion_Rate_%", ascending=False
    )
)


,month,Total_Contacts,Conversions,Conversion_Rate_%
5,mar,546,276,50.55
2,dec,182,89,48.90
9,sep,570,256,44.91
8,oct,717,315,43.93
0,apr,2631,539,20.49
1,aug,6176,655,10.61
4,jun,5318,559,10.51
7,nov,4100,416,10.15
3,jul,7169,648,9.04
6,may,13767,886,6.44


In [9]:
# Final KPI summary
total_contacts = len(df_clean)
conversions = (df_clean["y"] == "yes").sum()
non_conversions = (df_clean["y"] == "no").sum()
conversion_rate = conversions / total_contacts * 100

print("FINAL MARKETING CAMPAIGN SUMMARY")
print("-" * 38)
print("Total contacts:", total_contacts)
print("Successful conversions:", conversions)
print("Unsuccessful responses:", non_conversions)
print(f"Conversion rate: {conversion_rate:.2f}%")
print(f"Non-conversion rate: {100-conversion_rate:.2f}%")

# Export cleaned dataset, including age_group
df_clean.to_csv(
    "FUTURE_DS_03_Marketing_Cleaned.csv",
    index=False
)

print("\nCleaned dataset exported successfully!")


FINAL MARKETING CAMPAIGN SUMMARY
--------------------------------------
Total contacts: 41176
Successful conversions: 4639
Unsuccessful responses: 36537
Conversion rate: 11.27%
Non-conversion rate: 88.73%

Cleaned dataset exported successfully!
